In [5]:
import numpy as np
import pandas as pd
import tensorflow as tf

# Load the split datasets
X_train = pd.read_csv("X_train.csv")
X_dev = pd.read_csv("X_dev.csv")
X_test = pd.read_csv("X_test.csv")

y_train = pd.read_csv("y_train.csv").values.ravel()
y_dev = pd.read_csv("y_dev.csv").values.ravel()
y_test = pd.read_csv("y_test.csv").values.ravel()

print(f"Data Loaded: Train {X_train.shape}, Dev {X_dev.shape}, Test {X_test.shape}")

Data Loaded: Train (4922, 30), Dev (1055, 30), Test (1055, 30)


In [2]:
from tensorflow.keras.layers import Dense
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2


def build_baseline_model(input_dim):
    model = Sequential(
        [
            # Hidden Layer 1: 32 neurons, ReLU activation, L2 Regularization
            Dense(
                64,
                activation="relu",
                input_shape=(input_dim,),
                kernel_regularizer=l2(0.01),
            ),
            # Hidden Layer 2: 16 neurons, ReLU activation, L2 Regularization
            Dense(32, activation="relu", kernel_regularizer=l2(0.01)),
            # Output Layer: 1 neuron, Sigmoid activation (for binary classification)
            Dense(1, activation="sigmoid"),
        ]
    )

    # Compile with Adam optimizer and standard Binary Crossentropy loss
    model.compile(
        optimizer=Adam(learning_rate=0.005),
        loss="binary_crossentropy",
        metrics=["accuracy", tf.keras.metrics.Recall()],
    )
    return model


# Instantiate the model
input_dimension = X_train.shape[1]
model = build_baseline_model(input_dimension)
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │         1,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,097 (16.00 KB)

 Trainable params: 4,097 (16.00 KB)

 Non-trainable params: 0 (0.00 B)

In [3]:
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_dev, y_dev),
    epochs=50,
    batch_size=64,
    verbose=1,
)

Epoch 1/50
77/77 ━━━━━━━━━━━━━━━━━━━━ 4s 25ms/step - accuracy: 0.7879 - loss: 0.6941 - recall: 0.4564 - val_accuracy: 0.8095 - val_loss: 0.4839 - val_recall: 0.6299
Epoch 2/50
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7956 - loss: 0.4658 - recall: 0.5176 - val_accuracy: 0.8085 - val_loss: 0.4421 - val_recall: 0.5907
Epoch 3/50
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8015 - loss: 0.4498 - recall: 0.5291 - val_accuracy: 0.8085 - val_loss: 0.4460 - val_recall: 0.4448
Epoch 4/50
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7997 - loss: 0.4443 - recall: 0.5084 - val_accuracy: 0.8123 - val_loss: 0.4341 - val_recall: 0.6335
Epoch 5/50
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8039 - loss: 0.4422 - recall: 0.5344 - val_accuracy: 0.8171 - val_loss: 0.4356 - val_recall: 0.6512
Epoch 6/50
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7940 - loss: 0.4446 - recall: 0.5076 - val_accuracy: 0.7905 - val_loss: 0.4383 - val_recall: 0.3594
Epoch 7/50
77/7

**1.At first Iteration/run ,we observed:**

we achived 80.3 accuracy on train and 79.8 on dev

set ,so variance is accurate
but the model is underfit because only 80%

accuracy it is providing

the recall percentage is also 51 for train set

and 44 for dev set , so model underperforming ,

there is a bias

model was:

first hidden layer : 32(relu with l2)

second hidden layer : 16(relu with l2)

output layer : 1(sigmoid)

adam optimizer

**2.On the Second run,By doubling the neurons:**

train accuracy : 80.6 and dev accuracy is 81.4(increaded)

recal on train set is 50 and recall on dev set is about 53.7(increaded)

still there is high bias ,model is underperforming



In [6]:
import numpy as np
import tensorflow as tf
from sklearn.metrics import classification_report, f1_score
from tensorflow.keras.layers import Dense
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam

# 1. Make a deeper, more complex network (Andrew Ng's fix for High Bias)
complex_model = Sequential(
    [
        Dense(128, activation="relu", input_shape=(X_train.shape[1],)),
        Dense(64, activation="relu"),
        Dense(32, activation="relu"),
        Dense(1, activation="sigmoid"),
    ]
)

complex_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=[tf.keras.metrics.Precision(), tf.keras.metrics.Recall()],
)

# 2. Train the model
complex_model.fit(
    X_train, y_train, validation_data=(X_dev, y_dev), epochs=40, batch_size=64
)

# 3. Predict Raw Probabilities
dev_probs = complex_model.predict(X_dev).flatten()

# 4. Andrew Ng's Threshold Tweak: Lowering threshold to 0.35 to catch more churners
custom_threshold = 0.35
dev_preds_custom = (dev_probs >= custom_threshold).astype(int)

# 5. Evaluate using the exact metrics from the slides
print(f"--- Performance at Threshold {custom_threshold} ---")
print(classification_report(y_dev, dev_preds_custom))
print(f"Final F1-Score: {f1_score(y_dev, dev_preds_custom):.4f}")

Epoch 1/40


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


77/77 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step - loss: 0.4705 - precision: 0.6385 - recall_1: 0.3119 - val_loss: 0.4165 - val_precision: 0.6696 - val_recall_1: 0.5480
Epoch 2/40
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.4191 - precision: 0.6481 - recall_1: 0.5604 - val_loss: 0.4138 - val_precision: 0.6548 - val_recall_1: 0.5872
Epoch 3/40
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.4113 - precision: 0.6679 - recall_1: 0.5505 - val_loss: 0.4118 - val_precision: 0.6778 - val_recall_1: 0.5765
Epoch 4/40
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.4082 - precision: 0.6631 - recall_1: 0.5612 - val_loss: 0.4147 - val_precision: 0.6869 - val_recall_1: 0.5231
Epoch 5/40
77/77 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.4015 - precision: 0.6877 - recall_1: 0.5405 - val_loss: 0.4175 - val_precision: 0.6604 - val_recall_1: 0.6228
Epoch 6/40
77/77 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.3963 - precision: 0.6845 - recall_1: 0.5589 - val_loss: 0.4201 - val_precision: 0.7086 - val_recall_

In [7]:
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2

# 1. Maintain a powerful network, but add shields against Overfitting (High Variance)
regularized_model = Sequential(
    [
        # Input/Hidden Layer 1
        Dense(128, activation="relu", kernel_regularizer=l2(0.001)),
        Dropout(0.3),  # Randomly deactivates neurons to stop memorization
        # Hidden Layer 2
        Dense(64, activation="relu", kernel_regularizer=l2(0.001)),
        Dropout(0.3),
        # Hidden Layer 3
        Dense(32, activation="relu", kernel_regularizer=l2(0.001)),
        Dropout(0.2),
        # Output Layer
        Dense(1, activation="sigmoid"),
    ]
)

regularized_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=[tf.keras.metrics.Precision(), tf.keras.metrics.Recall()],
)

# 2. Add Early Stopping to stop training the second Val Loss stops improving
callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=8,  # Stop if it doesn't get better for 8 epochs
        restore_best_weights=True,  # Keeps your best model version
    )
]

# 3. Train the model
history_reg = regularized_model.fit(
    X_train,
    y_train,
    validation_data=(X_dev, y_dev),
    epochs=40,
    batch_size=64,
    callbacks=callbacks,
    verbose=1,
)

# 4. Re-run your prediction evaluation
dev_probs_reg = regularized_model.predict(X_dev).flatten()
dev_preds_reg = (dev_probs_reg >= 0.35).astype(int)

print(f"\n--- Performance at Threshold 0.35 (With Regularization) ---")
print(classification_report(y_dev, dev_preds_reg))
print(f"New F1-Score: {f1_score(y_dev, dev_preds_reg):.4f}")

Epoch 1/40
77/77 ━━━━━━━━━━━━━━━━━━━━ 10s 55ms/step - loss: 0.6663 - precision_1: 0.5262 - recall_2: 0.2072 - val_loss: 0.5767 - val_precision_1: 0.6364 - val_recall_2: 0.5480
Epoch 2/40
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.5800 - precision_1: 0.6262 - recall_2: 0.5008 - val_loss: 0.5450 - val_precision_1: 0.6386 - val_recall_2: 0.6477
Epoch 3/40
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.5449 - precision_1: 0.6278 - recall_2: 0.5275 - val_loss: 0.5241 - val_precision_1: 0.6504 - val_recall_2: 0.6157
Epoch 4/40
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.5295 - precision_1: 0.6456 - recall_2: 0.5237 - val_loss: 0.5067 - val_precision_1: 0.6822 - val_recall_2: 0.5730
Epoch 5/40
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.5070 - precision_1: 0.6506 - recall_2: 0.5382 - val_loss: 0.4945 - val_precision_1: 0.6767 - val_recall_2: 0.5587
Epoch 6/40
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.4984 - precision_1: 0.6514 - recall_2: 0.5229 - val_loss: 0.4873 - v

In [8]:
# Save your perfected model weights and architecture
regularized_model.save("final_churn_ann.keras")
print("Model saved successfully! Download 'final_churn_ann.keras' for your FastAPI app.")

Model saved successfully! Download 'final_churn_ann.keras' for your FastAPI app.
